#### Day 6: Machine Learning Model

we will work with 3 models: Logistic Regression, Decision Tree, and Random Forest.

In [3]:
import pandas as pd  
import joblib  # Import Joblib to load our saved preprocessor

X_train = pd.read_csv("../data/X_train.csv")  
X_test = pd.read_csv("../data/X_test.csv")  

y_train = pd.read_csv("../data/y_train.csv").squeeze("columns")  # Load training target
y_test = pd.read_csv("../data/y_test.csv").squeeze("columns")  # Load testing target

preprocessor = joblib.load("../models/preprocessor.pkl")  # Load fitted preprocessing pipeline

print("Training data:", X_train.shape)  
print("Testing data:", X_test.shape)  
print("Preprocessor loaded successfully!") 

#LOADING THE PREPROCESSOR , Because ensurng that our models receive features processed using the same rules 

Training data: (94987, 27)
Testing data: (24221, 27)
Preprocessor loaded successfully!


In [4]:

X_train_processed = preprocessor.transform(X_train)  # Transform training features using saved preprocessing

X_test_processed = preprocessor.transform(X_test)  # Transform testing features using the same preprocessing

print("Training data:", X_train_processed.shape)  
print("Testing data:", X_test_processed.shape) 


Training data: (94987, 103)
Testing data: (24221, 103)


Why are we using .transform() instead of .fit_transform()?
Because we already fitted our preprocessor on Day 5.
- fit_transform() → Learns preprocessing rules and applies them.
- transform() → Applies the previously learned rules.

#### Training logistic regression 

Logistic Regression is a classification algorithm used to predict categories.

In our Hotel Booking ML Project, it predicts:
- 0 → Booking will not be canceled.
- 1 → Booking will be canceled.
It can also estimate the probability of cancellation.

For example, if a booking has an estimated cancellation probability of 0.80, that means the model predicts an 80% cancellation risk.

In [8]:

from sklearn.linear_model import LogisticRegression  

lr_model = LogisticRegression(max_iter=1000, random_state=42)  
lr_model.fit(X_train_processed, y_train)  

print("Logistic Regression trained successfully!")  

#max_iter -> it allows upto 1000 iterations for the model to coverage


Logistic Regression trained successfully!


#### Evaluating logistic regression

1. accuracy -> tells the percentage of predictions the model got correct

In [10]:

from sklearn.metrics import accuracy_score  

y_pred_lr = lr_model.predict(X_test_processed)  # Predict cancellations for test data

lr_accuracy = accuracy_score(y_test, y_pred_lr)  # Compare predictions with actual results

print("Logistic Regression Accuracy:", round(lr_accuracy * 100, 2), "%")  # Display accuracy percentage

#round(lr_accuracy * 100, 2) -> means that roundingof the prctnage using 2 decimal points

Logistic Regression Accuracy: 81.14 %


#### classifcation report

calculating 
Precision: Of the bookings predicted as canceled, how many were actually canceled?
Recall: Of the bookings that were actually canceled, how many did our model identify?
F1-score: A balance between precision and recall.

In [11]:

from sklearn.metrics import classification_report 

print(classification_report(y_test, y_pred_lr))  # Display precision, recall, F1-score and support


              precision    recall  f1-score   support

           0       0.81      0.90      0.86     15147
           1       0.80      0.66      0.72      9074

    accuracy                           0.81     24221
   macro avg       0.81      0.78      0.79     24221
weighted avg       0.81      0.81      0.81     24221



Accuracy measures overall correctness, while precision, recall, and F1-score help evaluate how well the model predicts each class.

#### Confusion Matrix
Let's see the actual number of correct and incorrect predictions.

In [13]:

from sklearn.metrics import confusion_matrix  
cm_lr = confusion_matrix(y_test, y_pred_lr)  # Compare actual and predicted labels

print("Logistic Regression Confusion Matrix:")
print(cm_lr)


Logistic Regression Confusion Matrix:
[[13696  1451]
 [ 3116  5958]]


nterpretation:
- 13,696: Correctly predicted bookings that were not canceled.
- 5,958: Correctly identified canceled bookings.
- 1,451: Predicted cancellation, but the booking was not canceled.
- 3,116: Predicted no cancellation, but the booking was actually canceled.
Our model correctly classified 19,654 out of 24,221 bookings, giving approximately 81.14% accuracy.

The main weakness is the 3,116 missed cancellations. We'll want to reduce these while keeping false alarms manageable.

#### Training Decision Tree Classifier

A Decision Tree makes predictions by following a series of questions, such as whether a booking has a non-refundable deposit or a long lead time.


In [15]:

from sklearn.tree import DecisionTreeClassifier  

dt_model = DecisionTreeClassifier(max_depth=10, random_state=42) 

dt_model.fit(X_train_processed, y_train) 
print("Decision Tree trained successfully!") 

#max_depth controls how deep the tree can grow. Limiting its depth can help reduce overfitting, where a model memorizes training data rather than learning patterns that generalize.

Decision Tree trained successfully!


#### Evaluate Decision Tree

In [17]:

y_pred_dt = dt_model.predict(X_test_processed)  # Predict cancellations using Decision Tree

dt_accuracy = accuracy_score(y_test, y_pred_dt)  # Calculate Decision Tree accuracy

print("Decision Tree Accuracy:", round(dt_accuracy * 100, 2), "%")  
print("\nDecision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))  # Display precision, recall and F1-score


Decision Tree Accuracy: 82.65 %

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86     15147
           1       0.77      0.76      0.77      9074

    accuracy                           0.83     24221
   macro avg       0.82      0.81      0.81     24221
weighted avg       0.83      0.83      0.83     24221



By comapring lr and rf, accuracy has improved by 1.51 percentage points,
Cancellation recall increased from 66% to 76%.
Cancellation F1-score increased from 72% to 77%.
However, cancellation precision decreased slightly from 80% to 77%.

This means Decision Tree identifies more actual cancellations, although it also produces more false cancellation alarms.

#### Training Random Forest Classifier

random Forest combines predictions from many Decision Trees to make a final prediction.

In [18]:

from sklearn.ensemble import RandomForestClassifier  

rf_model = RandomForestClassifier(
    n_estimators=100,  # Create 100 decision trees
    max_depth=15,  # Limit the depth of each tree
    random_state=42,  # Make results reproducible
    n_jobs=-1  # Use available CPU cores for faster training
)

rf_model.fit(X_train_processed, y_train)  # Train Random Forest using training data

print("Random Forest trained successfully!") 

Random Forest trained successfully!


#### Evaluate Random Forest

In [19]:

y_pred_rf = rf_model.predict(X_test_processed)  # Predict cancellations using Random Forest

rf_accuracy = accuracy_score(y_test, y_pred_rf)  # Calculate Random Forest accuracy

print("Random Forest Accuracy:", round(rf_accuracy * 100, 2), "%") 
print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf)) 


Random Forest Accuracy: 84.24 %

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.94      0.88     15147
           1       0.87      0.69      0.77      9074

    accuracy                           0.84     24221
   macro avg       0.85      0.81      0.82     24221
weighted avg       0.85      0.84      0.84     24221



Discovered:

Random Forest has the highest accuracy (84.24%) and cancellation precision (87%).

However, there's something interesting:
- Decision Tree identifies approximately 76% of actual cancellations.
- Random Forest identifies approximately 69% of actual cancellations.
So although Random Forest is more accurate overall, Decision Tree is currently better at detecting cancellations.

This is an important finding for our project because a hotel may care more about identifying possible cancellations than simply achieving the highest accuracy.

We shouldn't declare Random Forest our final model yet. These are preliminary test results, and we still need proper validation and tuning.

#### Compare training and testing accuracy

checking whether any of our models might be overfitting.
Overfitting means a model performs very well on training data but struggles with unseen data.

In [20]:

models = {
    "Logistic Regression": lr_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model
}  # Store our three trained models

for name, model in models.items():  # Check each model
    train_acc = model.score(X_train_processed, y_train)  # Training accuracy
    test_acc = model.score(X_test_processed, y_test)  # Testing accuracy

    print(name)
    print("Training Accuracy:", round(train_acc * 100, 2), "%")
    print("Testing Accuracy:", round(test_acc * 100, 2), "%")
    print("Difference:", round((train_acc - test_acc) * 100, 2), "percentage points")
    print()


Logistic Regression
Training Accuracy: 81.04 %
Testing Accuracy: 81.14 %
Difference: -0.1 percentage points

Decision Tree
Training Accuracy: 84.29 %
Testing Accuracy: 82.65 %
Difference: 1.64 percentage points

Random Forest
Training Accuracy: 86.87 %
Testing Accuracy: 84.24 %
Difference: 2.63 percentage points



What the above result tells us

Logistic Regression shows very consistent training and testing performance. Decision Tree and Random Forest perform better on training data, which is normal, but their gaps suggest some overfitting may be present.

Random Forest still has the highest test accuracy at 84.24%, while Decision Tree has the highest cancellation recall at 76%.

Neither is automatically the best final model.

#### Create a Model Comparison Table

In [21]:

from sklearn.metrics import precision_score, recall_score, f1_score  

comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [lr_accuracy, dt_accuracy, rf_accuracy],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ]
})  # Create a table comparing all three models

comparison_df.round(4)  # Display results rounded to four decimal places


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8114,0.8042,0.6566,0.7229
1,Decision Tree,0.8265,0.7713,0.7631,0.7672
2,Random Forest,0.8424,0.8661,0.6853,0.7652


In [22]:

comparison_df.to_csv("../data/model_comparison.csv", index=False)  # Save model results as CSV

print("Model comparison saved successfully!")  # Confirm file saved


Model comparison saved successfully!


#### Prepare Group-Aware Cross-Validation
Before running cross-validation, we need to identify bookings with identical input features.

Why? Our dataset contains duplicate feature combinations. We don't want identical bookings appearing in both the training and validation folds because that could make our evaluation overly optimistic.

We'll use StratifiedGroupKFold, which tries to maintain the cancellation ratio while keeping identical feature groups together.

In [23]:

from sklearn.model_selection import StratifiedGroupKFold  # Import group-aware cross-validation

train_groups = X_train.groupby(
    X_train.columns.tolist(),
    dropna=False,
    sort=False
).ngroup()  # Give identical feature combinations the same group ID

cv = StratifiedGroupKFold(
    n_splits=3,  # Divide training data into 3 folds
    shuffle=True,  # Shuffle groups before splitting
    random_state=42  # Make splitting reproducible
)

print("Unique training groups:", train_groups.nunique())
print("Cross-validation prepared successfully!")


Unique training groups: 67876
Cross-validation prepared successfully!


n_splits=3 
The training dataset will be divided into three folds. Each fold gets a turn as validation data, while the other two are used for training.

We'll also ensure preprocessing is fitted separately within each training fold when we run cross-validation.

#### Create a Pipeline for Cross-Validation

We need to combine preprocessing and model training into a single pipeline.

This ensures that during cross-validation, preprocessing learns only from each fold's training data, not its validation data.

In [25]:

from sklearn.pipeline import Pipeline  # Import Pipeline
from sklearn.base import clone  # Import clone to create an unfitted copy

lr_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),  # Fresh preprocessing for each fold
    ("model", LogisticRegression(max_iter=1000, random_state=42))  # Logistic Regression model
])

print("Logistic Regression pipeline created successfully!")


Logistic Regression pipeline created successfully!


Why use clone(preprocessor)?
Our original preprocessor was already fitted on the full training dataset.

clone() creates a new, unfitted copy with the same settings. This allows cross-validation to fit preprocessing independently within each fold and prevents validation-data leakage.

Pipeline is used to combne multiple MLsteps into one workflow

clone , we already fitted our processor, clone(processor) creates a fresh, unfitted copyof it

this is used bcs durng cross validtion, we dont want our preprocessor to learn info from validtion data


("preprocessing", clone(preprocessor))	Converts categorical features and scales numerical features
("model", LogisticRegression(...))	Trains Logistic Regression to predict cancellations


Why do we need a pipeline?

During cross-validation, we'll divide our training data into three folds.

For each fold, the pipeline will:
1. Learn preprocessing rules from that fold's training portion.
2. Transform the training and validation portions using those rules.
3. Train Logistic Regression.
4. Evaluate predictions on the validation portion.
This helps us get a more reliable estimate of performance without preprocessing leakage.

#### Cross-Validation for Logistic Regression

In [28]:

from sklearn.model_selection import cross_val_score  # Import cross-validation function

lr_cv_scores = cross_val_score(
    lr_pipeline,  # Our Logistic Regression pipeline
    X_train,  # Original training features (before preprocessing)
    y_train,  # Actual training labels
    groups=train_groups,  # Keep identical feature groups together
    cv=cv,  # Use our 3-fold StratifiedGroupKFold
    scoring="f1",  # Evaluate F1-score for canceled bookings (class 1)
    n_jobs=-1  # Use available CPU cores
)

print("Logistic Regression CV F1-scores:", lr_cv_scores)
print("Average CV F1-score:", round(lr_cv_scores.mean(), 4))


#What does this code do?
#Our training data is divided into three folds. The model trains three times, each time using a different fold for validation.

#The pipeline also refits preprocessing within each training fold, preventing validation-data leakage.

Logistic Regression CV F1-scores: [0.7120384  0.70857686 0.71910006]
Average CV F1-score: 0.7132


these numbers are only an illustration, not your expected exact output.

Why scoring="f1"? Because our project needs a balance between identifying actual cancellations and avoiding false cancellation alarms.

Important: We're using only X_train and y_train here. Our test dataset is not involved in cross-validation.

#### Decision Tree Cross-Validation

In [35]:

dt_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),  # Create a fresh preprocessing step
    ("model", DecisionTreeClassifier(max_depth=10, random_state=42))  # Add Decision Tree
])

dt_cv_scores = cross_val_score(
    dt_pipeline,  # Decision Tree pipeline
    X_train,  # Original training features
    y_train,  # Training target
    groups=train_groups,  # Keep identical bookings in the same fold
    cv=cv,  # Use 3-fold group-aware cross-validation
    scoring="f1",  # Calculate cancellation F1-score
    n_jobs=-1  # Use available CPU cores
)

print("Decision Tree CV F1-scores:", dt_cv_scores)
print("Average CV F1-score:", round(dt_cv_scores.mean(), 4))


Decision Tree CV F1-scores: [0.76666812 0.76619568 0.77432147]
Average CV F1-score: 0.7691


#### Random Forest Cross-Validation

In [34]:

rf_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),  # Fresh preprocessing for each fold
    ("model", RandomForestClassifier(
        n_estimators=100,  # Build 100 decision trees
        max_depth=15,  # Limit tree depth
        random_state=42,  # Reproducible results
        n_jobs=-1  # Use available CPU cores
    ))
])

rf_cv_scores = cross_val_score(
    rf_pipeline,  # Random Forest pipeline
    X_train,  # Original training features
    y_train,  # Training target
    groups=train_groups,  # Keep identical feature groups together
    cv=cv,  # Use 3-fold group-aware cross-validation
    scoring="f1",  # Calculate cancellation F1-score
    n_jobs=1  # Avoid running multiple Random Forest fits simultaneously
)

print("Random Forest CV F1-scores:", rf_cv_scores)
print("Average CV F1-score:", round(rf_cv_scores.mean(), 4))


Random Forest CV F1-scores: [0.75810258 0.75520336 0.7640677 ]
Average CV F1-score: 0.7591


What do these results mean?
- Decision Tree currently performs best at identifying booking cancellations based on F1-score.
- Random Forest comes second, approximately 1 percentage point behind.
- Logistic Regression has the lowest F1-score of the three.
An important observation: Random Forest had the highest test accuracy (84.24%), but Decision Tree has the highest cross-validation F1-score.

This shows why we shouldn't select a model using accuracy alone.

#### Saving the results altogether

In [38]:

# Create a comparison table using cross-validation results
cv_comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "CV Mean F1": [
        lr_cv_scores.mean(),
        dt_cv_scores.mean(),
        rf_cv_scores.mean()
    ],
    "CV Std F1": [
        lr_cv_scores.std(),
        dt_cv_scores.std(),
        rf_cv_scores.std()
    ]
})

# Display the results
print(cv_comparison_df.round(4))

# Save the results as a CSV file
cv_comparison_df.to_csv("../data/cv_comparison.csv", index=False)

print("Cross-validation comparison saved successfully!")


                 Model  CV Mean F1  CV Std F1
0  Logistic Regression      0.7132     0.0044
1        Decision Tree      0.7691     0.0037
2        Random Forest      0.7591     0.0037
Cross-validation comparison saved successfully!



## Model Training and Evaluation Summary

Three machine learning models were trained to predict hotel booking cancellations:

1. Logistic Regression
2. Decision Tree
3. Random Forest

### Cross-Validation Results

| Model | Average CV F1-score |
|---|---|
| Logistic Regression | 0.7132 |
| Decision Tree | 0.7691 |
| Random Forest | 0.7591 |

### Key Findings

- Decision Tree achieved the highest cross-validation F1-score (76.91%).
- Random Forest achieved the highest test accuracy (84.24%).
- All models were evaluated using 3-fold Stratified Group Cross-Validation.
- Group-aware splitting was used to prevent identical feature combinations from appearing in both training and validation folds.
- Preprocessing was fitted separately within each cross-validation fold to prevent data leakage.

### Conclusion

Decision Tree currently performs best based on cross-validation F1-score for cancellation prediction.

Further hyperparameter tuning will be performed before selecting the final model.
